# Adaptive PDE discretizations on cartesian grids 
## Volume : Divergence form PDEs
## Part : Linear elasticity
## Chapter : The wave equation, optimized

We present a numerical scheme for the elastic wave equation, which is closely related to the one of the [elastic energy](ElasticEnergy.ipynb) and [elastic wave](ElasticWave.ipynb) notebooks. In fact, it is equivalent in the case of a constant hooke tensor over the domain, and of periodic boundary conditions. The differences are as follows:
* (Memory efficiency) The implementation is designed to use much less memory, and thus to allow addressing larger test cases. In particular, this implementation does involve computing the evolution operators sparse matrices.
* (Second and fourth order) A fourth order variant of the scheme is implemented, available on demand.
* (Boundary conditions) In addition to periodic boundary conditions, a crude implementation of absorbing boundary conditions s proposed, via a damping layer. 

This implementation has also a few downfalls w.r.t the one of the previous notebooks [elastic energy](ElasticEnergy.ipynb) and [elastic wave](ElasticWave.ipynb):
* (GPU only) The scheme is implemented via custom cuda kernels. There is no CPU variant, sorry.
* (Less flexible and pedagogical) The previous implementation was fully exposed in the notebooks. In contrast, most of this scheme is hard coded in the agd library.
* (Approximate energy conservation) The previous implemetation was based a symplectic operator of a discrete Hamiltonian, which guarantees the conservation of suitable perturbation of the Hamiltonian. In contrast, the present implementation fails this property, more on this below.

**Divergence form, and non-divergence form PDEs**
In order to illustrate the difference between the two implementations of the elastic wave equation, let us first consider a scalar PDE operator. Let $A$ be a smooth field of symmetric positive definite matrices over a domain $\Omega$, and $u$ a smooth scalar function. Then one has the identity
$$
    \mathrm{div}(A \nabla u) = \mathrm{tr}(A \nabla^2 u) + <\mathrm{div} A, \nabla u>,
$$
where the divergence of $A$ is row-wise (or column-wise, which makes no difference since $A$ is symmetric).

While the two sides of the previous equation are identical at the continuous level, their natural discretizations substantially differ.
* (Divergence form) The operator $\mathrm{div}(A \nabla u)$ is best discretized by approximating the Dirichlet energy 
$$
    \mathcal E(u) := \frac 1 2 \int_\Omega <\nabla u, A \nabla u>,
$$
and then differentiating $\mathcal E$ w.r.t the unknown $u$. See the notebooks on [elliptic](Elliptic.ipynb) or [anisotropic diffusion](AnisotropicDiffusion.ipynb) equations.
* (Non-divergence form) The operator $\mathrm{tr}(A \nabla^2 u) + <\omega, \nabla u>$, for an arbitrary choice of vector field $\omega$, can be discretized as a generic [non-divergence form second order linear operator](../Notebooks_NonDiv/LinearMonotoneSchemes2D.ipynb).

The guarantees associated with these two discretizations are quite distinct: the divergence form discretization benefits from energy estimates, whereas the non-divergence form discretization enjoys the maximum principle.

For vector equations, such as the elastic wave equation, there is no such thing as a maximum principle, hence it is natural to adopt the divergence form approach so as to take advantage of the associated estimates. Unfortunately, the divergence form approach requires assembling the sparse operator matrix, which can be excessively costly in our application.
The implementation presented in this notebook relies on the non-divergence form approach, whereas the previous notebooks used the divergence form approach [elliptic](Elliptic.ipynb).

## 0. Importing the required libraries

In [ ]:
from agd.Eikonal.HFM_CUDA.HookeWave import HookeWave
from agd import LinearParallel as lp
from agd import FiniteDifferences as fd
from agd.Metrics.Seismic import Hooke
from agd.ODE.hamiltonian import QuadraticHamiltonian
from agd import AutomaticDifferentiation as ad
from agd import Domain
from agd.Plotting import savefig,quiver; #savefig.dirName = 'Images/ElasticityDirichlet'
norm_infinity = ad.Optimization.norm_infinity